In [103]:
from typing import TypedDict, Annotated, Literal
import sqlite3
import requests

from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_ollama import ChatOllama
from langchain_core.messages import BaseMessage, SystemMessage,HumanMessage
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_ollama import OllamaEmbeddings
load_dotenv()
import os

In [104]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0,
    base_url="http://172.31.0.1:11434"
)

subgraph_llm = ChatOllama(
    model="llama3.2",
    temperature=0,
    base_url="http://172.31.0.1:11434"
)

In [105]:
class Substate(TypedDict):
    input_text: str
    translated_text: str

In [106]:
def translate_text(state: Substate):

    prompt = f"""
    Translate the following text to Hindi.
    Keep it natural and do not add extra content:

    {state['input_text']}
    """.strip()

    translated = subgraph_llm.invoke(prompt).content

    return {
        "translated_text": translated
    }

In [107]:
subgraph = StateGraph(Substate)

subgraph.add_node("translate_text", translate_text)

subgraph.add_edge(START, "translate_text")
subgraph.add_edge("translate_text", END)

subgraph_chat = subgraph.compile()

In [108]:
class parentstate(TypedDict):
    question: str
    answer_eng: str
    answer_hindi: str

In [109]:
def generate_answer(state: parentstate):

    answer = llm.invoke(
        f"""
        Answer the following question in English.
        Keep the answer within 100 words.

        Question: {state['question']}
        """.strip()
    )

    return {
        "answer_eng": answer.content
    }

In [110]:
def translate_parent(state: parentstate):

    result = subgraph_chat.invoke({
        "input_text": state["answer_eng"]
    })

    return {
        "answer_hindi": result["translated_text"]
    }

In [111]:
parent = StateGraph(parentstate)

parent.add_node("generate_answer", generate_answer)

# IMPORTANT:
# Use translate_parent here, NOT translate_text
parent.add_node("translate_text", translate_parent)

parent.add_edge(START, "generate_answer")
parent.add_edge("generate_answer", "translate_text")
parent.add_edge("translate_text", END)

parent_chat = parent.compile()

In [112]:
result = parent_chat.invoke({
    "question": "What quantum physics is and how it works in 100 words only?"
})

In [ ]:
print("English Answer:")
print(result["answer_eng"])

print("\nHindi Answer:")
print(result["answer_hindi"])